### Block 0 – Install dependencies


In [ ]:
!pip install -q openai datasets huggingface_hub scikit-learn tqdm pandas


### Block 1 – Imports, API keys, and OpenAI client

In [ ]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfFolder
from openai import OpenAI
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

# OpenAI API key
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (sk-...): ")
client = OpenAI()

# Hugging Face token for the gated dataset
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf-...): ")


OpenAI API key (sk-...): ··········
Hugging Face token (hf-...): ··········


### Block 2 – Load FiQA-SA test split (235 examples)

In [ ]:
DATASET_ID = "TheFinAI/flare-fiqasa"

# Load all splits using the token
ds_all = load_dataset(DATASET_ID, token=hf_token)

print("Available splits and sizes:")
for split_name in ds_all.keys():
    print(f"  {split_name}: {len(ds_all[split_name])}")

# Use the official test split (235 examples)
EVAL_SPLIT = "test"
dataset = ds_all[EVAL_SPLIT]

print("\nUsing split:", EVAL_SPLIT)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])

# Hard check to ensure we are really using 235 examples
assert len(dataset) == 235, f"Expected 235 test examples, got {len(dataset)}"


Available splits and sizes:
  train: 750
  test: 235
  valid: 188

Using split: test
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 3 – Label set and normalization helper

In [ ]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map free-form model output to one of the three labels.
    If no label keyword is found, default to 'neutral'.
    """
    text = (raw or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    return "neutral"


### Block 4 – o3-mini sentiment classifier

In [ ]:
MODEL_NAME_O3 = "o3-mini"

O3_SYSTEM_PROMPT = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog, "
    "classify its overall sentiment from the perspective of an investor as "
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_o3_mini(sentence: str) -> str:
    """
    Call o3-mini once and return a normalized 3-way label.
    Uses the same LABELS and normalize_prediction as other models.
    """
    completion = client.chat.completions.create(
        model=MODEL_NAME_O3,
        messages=[
            {"role": "system", "content": O3_SYSTEM_PROMPT},
            {"role": "user", "content": sentence},
        ],
        # o3-mini requires max_completion_tokens (not max_tokens)
        max_completion_tokens=16,
        # do not set temperature (o3-mini only supports the default)
    )
    raw = completion.choices[0].message.content or ""
    return normalize_prediction(raw)


### Block 5 – Run evaluation on the 235 test examples

In [ ]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_NAME_O3} on {len(dataset)} FiQA-SA test examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_o3_mini(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: o3-mini on 235 FiQA-SA test examples...



100%|██████████| 235/235 [03:10<00:00,  1.24it/s]

Total examples: 235
Got predictions for: 235


### Block 6 – Metrics and error analysis

In [ ]:
# Overall accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Build a DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.0766

Classification report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        76
     neutral       0.08      1.00      0.14        18
    positive       0.00      0.00      0.00       141

    accuracy                           0.08       235
   macro avg       0.03      0.33      0.05       235
weighted avg       0.01      0.08      0.01       235


First 10 predictions:
                                                                                                                            text     gold    pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative neutral
                                                                         Legal & General share price: Finance chief to step down negative neutral
                                                                 Kingfisher share price slides on cost to im

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


####

On the FiQA-SA test split of 235 short financial texts, o3-mini achieves a very low overall accuracy of 0.0766. The classification report shows that recall for the negative and positive classes is 0, while recall for neutral is 1.0. In practice, this means that under the current zero-shot prompt, the model is predicting almost every example as neutral.

This behaviour indicates that the model is extremely conservative on this dataset: many tweets and headlines that FiQA-SA annotators label as clearly negative or positive (from an investor’s perspective) are treated by o3-mini as neutral factual statements. As a result, the predictions effectively collapse into a single majority class, and the measured accuracy ends up close to the proportion of neutral labels in the test set (18/235 ≈ 0.0766).

Therefore, this score should be interpreted with caution. It reflects a strong mismatch between the dataset’s annotation scheme and the model’s default notion of “sentiment” on short financial microtext, rather than a pure limitation in o3-mini’s language understanding. A more aggressive or example-guided prompt (few-shot) would likely be needed to push the model away from its neutral default and better align it with FiQA-SA’s labeling criteria.